In [ ]:
!pip install -U "transformers>=4.38.0" datasets accelerate evaluate


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset

from google.colab import files

# Transformers components
from transformers import (
    AutoTokenizer, AutoModel,
    TrainingArguments, Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    TrainerCallback
)

# Performance evaluation components
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report, f1_score

# --------------------------------------------
# 0. Basic configuration
# --------------------------------------------
SEED = 42
MODEL_NAME = "klue/roberta-base"
SAVE_PATH = "/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------
# 1. Mount Google Drive
# --------------------------------------------
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Configure the save path
SAVE_PATH = "/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/"
os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(os.path.join(SAVE_PATH, "logs"), exist_ok=True)

# Check CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -----------------------------------
# 1) Load the dataset
# -----------------------------------
print("Upload a CSV or XLSX file.")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Check the file extension and load the data
if filename.endswith('.csv'):
    data = pd.read_csv(filename)
elif filename.endswith('.xlsx'):
    data = pd.read_excel(filename)
else:
    raise ValueError("Unsupported file format. Upload a CSV or XLSX file.")

# -----------------------------------
# 2) Data preprocessing
# -----------------------------------
attribute_columns = [
    'INFOSRC', 'INFOSRC_CODE_2', 'ART_DATE', 'ART_PROVIDER', 'ART_CATEGORY1','ART_CATEGORY2', 'ART_CATEGORY3',
    'ART_TAG_1', 'ART_TAG_2', 'ART_TAG_3', 'SNT_TAG_1', 'SNT_TAG_2','SNT_TAG_3', 'ART_HEADLINE', 'ART_BYLINE'
]

required_columns = ["STN_CONTENT", "OPN"] + attribute_columns

missing_columns = [col for col in required_columns if col not in data.columns]

if missing_columns:
    raise ValueError(f"The following required columns are missing: {missing_columns}")

# --------------------------------------------
# 4. Handle missing values and create the attribute text
# --------------------------------------------
data = data.copy()

data["STN_CONTENT"] = data["STN_CONTENT"].fillna("").astype(str)
data["OPN"] = data["OPN"].fillna("").astype(str)

for col in attribute_columns:
    data[col] = data[col].fillna("").astype(str)

# Remove rows with empty text or labels
data = data[
    (data["STN_CONTENT"].str.strip() != "") &
    (data["OPN"].str.strip() != "")
].reset_index(drop=True)

if len(data) == 0:
    raise ValueError("No valid training data are available. Check the STN_CONTENT and OPN columns.")

data["attributes_text"] = data[attribute_columns].agg(" ".join, axis=1)

# --------------------------------------------
# 5. Label encoding
# --------------------------------------------
label_encoder = LabelEncoder()
data["label"] = label_encoder.fit_transform(data["OPN"])

label_names = list(label_encoder.classes_)
num_labels = len(label_names)

print("Labels:", label_names)
print("Number of labels:", num_labels)
print("Label distribution:")
print(data["OPN"].value_counts())

if num_labels < 2:
    raise ValueError("Fewer than two labels are available; the classification model cannot be trained.")

# --------------------------------------------
# 6. Shuffle the data and convert them to a Hugging Face Dataset
# --------------------------------------------
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)

hf_dataset = Dataset.from_pandas(
    data[["STN_CONTENT", "attributes_text", "label"]],
    preserve_index=False
)

# First split the data 80:20 (train+eval:test),
# then split the remaining 80% at a 75:25 ratio (train:eval).
# Final ratio: 60:20:20 (train:eval:test).

split_data = hf_dataset.train_test_split(test_size=0.2, seed=SEED)
test_dataset_hf = split_data['test']         # 20%
remaining_dataset = split_data['train']      # 80%

split_data2  = remaining_dataset.train_test_split(test_size=0.25, seed=42)  # 25% of 80% = 20% of the full dataset
train_dataset_hf = split_data2['train']      # Final 60%
eval_dataset_hf  = split_data2['test']       # Final 20%

print(f"Train dataset size: {len(train_dataset_hf)}")
print(f"Eval dataset size:  {len(eval_dataset_hf)}")
print(f"Test dataset size:  {len(test_dataset_hf)}")

# -----------------------------------
# 7. Tokenizer
# -----------------------------------
MODEL_NAME = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(example):
    encoding1 = tokenizer(example['attributes_text'], padding='max_length', truncation=True, max_length=256)
    encoding2 = tokenizer(example['STN_CONTENT'], padding='max_length', truncation=True, max_length=256)
    return {
        'input_ids1': encoding1['input_ids'],
        'attention_mask1': encoding1['attention_mask'],
        'input_ids2': encoding2['input_ids'],
        'attention_mask2': encoding2['attention_mask'],
        'label': example['label']
    }
# -----------------------------------
# 8. Convert the datasets
# -----------------------------------
train_dataset = train_dataset_hf.map(tokenize_function, batched=True)
eval_dataset = eval_dataset_hf.map(tokenize_function, batched=True)
test_dataset = test_dataset_hf.map(tokenize_function, batched=True)

# Remove the original text columns
columns_to_remove = ["STN_CONTENT", "attributes_text"]
train_dataset = train_dataset.remove_columns(columns_to_remove)
eval_dataset = eval_dataset.remove_columns(columns_to_remove)
test_dataset = test_dataset.remove_columns(columns_to_remove)

# -----------------------------------
# 6) Define the KLUE-RoBERTa-based MoE model
# -----------------------------------
class MoEBertModel(nn.Module):
    def __init__(self, num_labels):
        super(MoEBertModel, self).__init__()
        self.num_labels = num_labels
        self.hidden_size = 768

        # Expert 1 (attribute-text input)
        self.bert_expert1 = AutoModel.from_pretrained('klue/roberta-base')

        # Expert 2 (quotation-text input)
        self.bert_expert2 = AutoModel.from_pretrained('klue/roberta-base')

        self.classifier1 = nn.Linear(self.hidden_size, num_labels)
        self.classifier2 = nn.Linear(self.hidden_size, num_labels)

        # Freeze some of the initial layers (e.g., the first six layers)
        for param in self.bert_expert1.encoder.layer[:6].parameters():
            param.requires_grad = False
        for param in self.bert_expert2.encoder.layer[:6].parameters():
            param.requires_grad = False

        # Gating network (determines how to combine the two experts' pooled outputs)
        self.gating_network = nn.Sequential(
            nn.Linear(self.hidden_size * 2, self.hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_size, 2),
            nn.Softmax(dim=1))

        # Dropout and final classifier
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.hidden_size, self.num_labels)

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, labels=None):

        # Expert 1
        encoding1 = self.bert_expert1(input_ids=input_ids1, attention_mask=attention_mask1).pooler_output

        # Expert 2
        encoding2 = self.bert_expert2(input_ids=input_ids2, attention_mask=attention_mask2).pooler_output

        # Gating network: concatenate the two experts' pooled outputs to determine their weights
        logits1 = self.classifier1(encoding1)
        logits2 = self.classifier2(encoding2)

        combined = torch.cat((encoding1, encoding2), dim=1)  # [batch_size, hidden_size * 2]
        gating_weights = self.gating_network(combined)  # [batch_size, 2]

        # Stack the expert outputs
        expert_outputs = torch.stack([logits1, logits2], dim=1)  # [batch_size, 2, hidden_size]

        # Compute the weighted sum of expert outputs using the gating weights
        gated_output = torch.einsum('bi,bih->bh', gating_weights, expert_outputs)  # [batch_size, hidden_size]

        # Calculate the loss
        loss = None
        if labels is not None:
            criterion = nn.CrossEntropyLoss()
            loss = criterion(gated_output, labels)

        if loss is not None:
            return {
                "loss": loss,
                "logits": gated_output
            }

        return {
            "logits": gated_output
        }

# Initialize the model (must be done before initializing Trainer)
model = MoEBertModel(num_labels=num_labels).to(device)
print("The model was created successfully.")

# -----------------------------------
# 9. Evaluation metrics function
# -----------------------------------
def compute_metrics(eval_pred):
    labels = eval_pred.label_ids
    preds = np.argmax(eval_pred.predictions, axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    return {'accuracy': acc,'macro_f1': f1,'macro_precision': precision,'macro_recall': recall}

# -----------------------------------
# 9) Configure Trainer
# -----------------------------------
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/10_Models/3-1_KLUE-MoE',
    num_train_epochs=100,
    per_device_train_batch_size=64, # Increased batch size
    per_device_eval_batch_size=64,
    report_to="none",  # Prevent Weights & Biases from launching automatically
    learning_rate=1e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),  # Enable mixed precision to improve speed

    # Log, evaluate, and save checkpoints after each epoch
    logging_strategy='epoch',
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Log directory
    logging_dir='/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/logs',
    remove_unused_columns=False
)

# Early-stopping callback
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5,
    early_stopping_threshold=0.001
)

# -----------------------------------
# 10) Use a custom data collator
# -----------------------------------
def custom_data_collator(batch):
    return {
        'input_ids1': torch.stack([torch.tensor(item['input_ids1']) for item in batch]),
        'attention_mask1': torch.stack([torch.tensor(item['attention_mask1']) for item in batch]),
        'input_ids2': torch.stack([torch.tensor(item['input_ids2']) for item in batch]),
        'attention_mask2': torch.stack([torch.tensor(item['attention_mask2']) for item in batch]),
        'labels': torch.tensor([item['label'] for item in batch], dtype=torch.long),
    }

# -----------------------------------
# 11) Define a callback to evaluate the training set after every epoch
# -----------------------------------
class TrainDatasetMetricsCallback(TrainerCallback):
    """
    After every epoch, this example callback uses the training dataset to
    calculate additional metrics (e.g., accuracy and loss)
    """
    def __init__(self, train_dataset):
        super().__init__()
        self.train_dataset = train_dataset
        self.trainer_ref = None

    def on_train_begin(self, args, state, control, **kwargs):
        # Store the Trainer object when training begins
        if "trainer" in kwargs:
            self.trainer_ref = kwargs["trainer"]

    def on_epoch_end(self, args, state, control, **kwargs):
        # Evaluate train_dataset at the end of every epoch
        if self.trainer_ref is not None:
            train_metrics = self.trainer_ref.evaluate(
                eval_dataset=self.train_dataset,
                metric_key_prefix="train"
            )
            train_acc  = train_metrics.get("train_accuracy", None)
            train_loss = train_metrics.get("train_loss", None)
            print(f"[Epoch {int(state.epoch)}] Train Accuracy: {train_acc:.4f}, Train Loss: {train_loss:.4f}")
        return control

# Create the callback instance
train_metrics_callback = TrainDatasetMetricsCallback(train_dataset)

# -----------------------------------
# 12) Create the Trainer object
# -----------------------------------
# Use custom_data_collator instead of Trainer's default data_collator
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=custom_data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,   # Calculate validation-set metrics
    callbacks=[early_stopping_callback, train_metrics_callback]
)

# -----------------------------------
# 13) Train the model
# -----------------------------------
trainer.train()

# -----------------------------------
# 14) Define the evaluation function
# -----------------------------------
def evaluate_and_report(trainer, dataset, dataset_name, label_names):
    predictions = trainer.predict(dataset)
    preds = predictions.predictions.argmax(-1)
    labels = predictions.label_ids

    acc = accuracy_score(labels, preds)
    f1_w = f1_score(labels, preds, average='weighted')
    print(f"\n===== [{dataset_name}] =====")
    print(f"{dataset_name} Accuracy: {acc:.4f}")
    print(f"{dataset_name} Weighted-F1: {f1_w:.4f}")

    # The label_names parameter can specify the label order
    print(classification_report(labels, preds, target_names=label_names))

# Actual label names from label_encoder.classes_ (e.g., ['Sell', 'Neutral', 'Buy'])
label_names = list(label_encoder.classes_)
print("Label names:", label_names)

# -----------------------------------
# 15) Final evaluation (train/validation/test)
# -----------------------------------
# (A) Evaluate the training set
evaluate_and_report(trainer, train_dataset, "Train Set", label_names)

# (B) Evaluate the validation set
eval_results = trainer.evaluate()  # eval_dataset=eval_dataset
print(f"\n[Validation Set] eval_loss: {eval_results['eval_loss']:.4f}")
evaluate_and_report(trainer, eval_dataset, "Validation Set", label_names)

# (C) Evaluate the test set
evaluate_and_report(trainer, test_dataset, "Test Set", label_names)

# -----------------------------------
# 16) Save the model and its weights
# -----------------------------------
torch.save(model, '/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/best_model.pt')
torch.save(model.state_dict(), '/content/drive/MyDrive/10_Models/3-1_KLUE-MoE/best_model_weights.pt')
print("The model and its weights have been saved.")
